# 03. Classifier Stress Test

Run the classifier across all downloaded bills.
Goal: find `unknown` nodes and false positives to drive pattern improvements in `classify_bill.py`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, "..")

from bill_tree import normalize_bill
from financial_classifier.classify_bill import DOLLAR, PRIMARY_LABELS, build_financial_df, check_coverage, classify_text

## Load and classify all bills

One version per bill (first downloaded). Adjust `xmls[0]` to target a specific version.

In [2]:
BILLS_DIR = Path("../bills")

rows = []
for bill_dir in sorted(BILLS_DIR.iterdir()):
    if not bill_dir.is_dir():
        continue
    xmls = sorted(bill_dir.glob("*.xml"))
    if not xmls:
        continue
    try:
        tree = normalize_bill(xmls[0])  # first version only
    except Exception as e:
        print(f"SKIP {bill_dir.name}: {e}")
        continue
    for n in tree.nodes:
        if not DOLLAR.search(n.body_text or ""):
            continue
        rows.append(
            {
                "bill": bill_dir.name,
                "label": classify_text(n.body_text),
                "path": " > ".join(n.display_path[-2:]) if n.display_path else "",
                "preview": (n.body_text or "")[:120],
                "body_text": n.body_text or "",
            }
        )

df = pd.DataFrame(rows)
print(f"{len(df)} dollar-amount nodes across {df['bill'].nunique()} bills")
df["label"].value_counts()

273 dollar-amount nodes across 2 bills


label
unknown          132
appropriation    116
fee                7
restriction        5
cap                5
directive          4
transfer           4
Name: count, dtype: int64

In [3]:
for _, row in df[df["label"] == "cap"].iterrows():
    print(row["bill"], "|", row["path"])
    print()
    print(row["body_text"])
    print("---")

118-hr-4366 | Administrative provisions > sec. 119

Notwithstanding any other provision of law, funds made available in this title for operation and maintenance of family housing shall be the exclusive source of funds for repair and maintenance of all family housing units, including general or flag officer quarters: Provided, That not more than $15,000 per unit may be spent annually for the maintenance and repair of any general or flag officer quarters without 30 days prior notification, or 14 days for a notification provided in an electronic medium pursuant to sections 480 and 2883 of title 10, United States Code, to the Committees on Appropriations of both Houses of Congress, except that an after-the-fact notification shall be submitted if the limitation is exceeded solely due to costs associated with environmental remediation that could not be reasonably anticipated at the time of the budget submission: Provided further, That the Under Secretary of Defense (Comptroller) is to report

## Label distribution by bill

In [4]:
df.pivot_table(
    index="bill",
    columns="label",
    values="path",
    aggfunc="count",
    fill_value=0,
)

label,appropriation,cap,directive,fee,restriction,transfer,unknown
bill,,,,,,,
118-hr-4366,51,2,4,0,5,4,1
119-hr-1,65,3,0,7,0,0,131


## Coverage check

Verify that `build_financial_df` captures every dollar-amount node — no silent drops.

In [5]:
BILLS_DIR = Path("../bills")
for bill_dir in sorted(BILLS_DIR.iterdir()):
    if not bill_dir.is_dir():
        continue
    xmls = sorted(bill_dir.glob("*.xml"))
    if not xmls:
        continue
    try:
        tree = normalize_bill(xmls[0])
    except Exception as e:
        print(f"SKIP {bill_dir.name}: {e}")
        continue
    df_fin = build_financial_df(tree)
    print(f"{bill_dir.name}:", end=" ")
    check_coverage(df_fin, tree)

118-hr-4366: ✓  All 67 dollar-amount nodes represented (103 rows)
119-hr-1: ✓  All 199 dollar-amount nodes represented (221 rows)


## Unknowns

These are the nodes that need new patterns in `classify_bill.py`.

In [6]:
unknowns = df[df["label"] == "unknown"][["bill", "path", "preview", "body_text"]].reset_index(drop=True)
print(f"{len(unknowns)} unknown nodes")
unknowns[["bill", "path", "preview"]]

132 unknown nodes


,bill,path,preview
0,118-hr-4366,GENERAL PROVISIONS > sec. 418,$0.
1,119-hr-1,sec. 10101 > (a) Reference price,(a)Reference price Section 1111(19) of the Agr...
2,119-hr-1,sec. 10101 > (g) Payment limitations,(g)Payment limitations Section 1001 of the Foo...
3,119-hr-1,sec. 10101 > (i) Marketing loans,(i)Marketing loans(1)Availability of nonrecour...
4,119-hr-1,sec. 10101 > (o) Implementation,(o)Implementation Section 1614(c) of the Agric...
...,...,...,...
127,119-hr-1,sec. 112206 > (a) Earned income tax credit cer...,(a)Earned income tax credit certification prog...
128,119-hr-1,sec. 112206 > (b) Task force to design a priva...,(b)Task force to design a private data bouncin...
129,119-hr-1,sec. 112207 > (b) Appropriation for task force...,(b)Appropriation for task force to design a be...
130,119-hr-1,sec. 112210 > (a) In general,"(a)In general Paragraphs(1),(2),(3),(4), and(5..."


### Group by opening text

Frequent openers are the highest-priority patterns to add.

In [7]:
# Group unknown nodes by opening text to find repeated patterns worth adding rules for
unknowns["opener"] = unknowns["body_text"].str[:80]
unknowns["opener"].value_counts().head(30)

opener
(c)Subsequent adjustment Beginning in fiscal year 2026 and each fiscal year ther    7
(b)Fee specified(1)Initial amount The amount specified in this subsection for fi    7
(b)Initial amount For purposes of this subsection, the amount specified in this     4
$0.                                                                                 1
(a)Reference price Section 1111(19) of the Agricultural Act of 2014 (7 U.S.C. 90    1
(g)Payment limitations Section 1001 of the Food Security Act of 1985 (7 U.S.C. 1    1
(i)Marketing loans(1)Availability of nonrecourse marketing assistance loans for     1
(o)Implementation Section 1614(c) of the Agricultural Act of 2014 (7 U.S.C. 9097    1
(p)Livestock safety net updates(1)In general Section 1501(b) of the Agricultural    1
(v)Program compliance and integrity Section 515(l)(2) of the Federal Crop Insura    1
(w)Reviews, compliance, and integrity Section 516(b)(2)(C)(i) of the Federal Cro    1
(a)Grassroots source water protection program S

### Inspect full text of a specific unknown

In [8]:
# Read full text of a specific unknown row for pattern investigation
# Change idx to the row number you want to inspect
for _, row in unknowns.iloc[::5].iterrows():
    print(row["bill"], "|", row["path"])
    print()
    print(row["body_text"])
    print("---")

118-hr-4366 | GENERAL PROVISIONS > sec. 418

$0.
---
119-hr-1 | sec. 10101 > (p) Livestock safety net updates

(p)Livestock safety net updates(1)In general Section 1501(b) of the Agricultural Act of 2014 (7 U.S.C. 9081(b)) is amended—(A)by amending paragraph(2) to read as follows:(2)Payment rates(A)Losses due to predation Indemnity payments to an eligible producer on a farm under paragraph(1)(A) shall be made at a rate of 100 percent of the market value of the affected livestock on the applicable date, as determined by the Secretary.(B)Losses due to adverse weather or disease Indemnity payments to an eligible producer on a farm under subparagraph(B) or(C) of paragraph(1) shall be made at a rate of 75 percent of the market value of the affected livestock on the applicable date, as determined by the Secretary.(C)Determination of market value In determining the market value described in subparagraphs(A) and(B), the Secretary may consider the ability of eligible producers to document regio

## False positive check

Spot-check `PRIMARY_LABELS` nodes — especially useful once non-appropriations bills are in the mix.

In [9]:
# Spot-check PRIMARY_LABELS nodes in non-appropriations bills
# Look for nodes labelled appropriation/transfer/rescission that shouldn't be
primary = df[df["label"].isin(PRIMARY_LABELS)]
for _, row in primary.sample(min(15, len(primary)), random_state=42).iterrows():
    print(f"[{row['bill']}] [{row['label']}]")
    print(row["body_text"][:200])
    print()

[118-hr-4366] [appropriation]
For military and naval insurance, national service life insurance, servicemen's indemnities, service-disabled veterans insurance, and veterans mortgage life insurance as authorized by chapters 19 and 

[119-hr-1] [appropriation]
(a)Appropriation In addition to amounts otherwise available, there is appropriated to the Office of Refugee Resettlement for fiscal year 2025, out of any money in the Treasury not otherwise appropriat

[119-hr-1] [appropriation]
(d)CBP vehicles In addition to amounts otherwise available, there is appropriated to the Commissioner of U.S. Customs and Border Protection for fiscal year 2025, out of any money in the Treasury not o

[119-hr-1] [appropriation]
(a)Appropriations In addition to amounts otherwise available, there are appropriated to the Secretary of Defense for fiscal year 2025, out of any money in the Treasury not otherwise appropriated, to r

[118-hr-4366] [appropriation]
For grants to assist States to acquire or construct